# Analiza emocjonalna tekstów kultury z użyciem modeli BERT i GPT

Projekt porównuje jakość klasyfikacji emocji na zbiorze **GoEmotions** dla dwóch rodzin modeli:

- **Modele typu BERT (fine-tuning):** DeBERTa, RoBERTa, ALBERT — trenowane jako klasyfikatory 28 klas.
- **Modele typu GPT (zero-shot):** Llama 3 i Gemma 2 — odpowiadają tekstem, który mapujemy na te same 28 etykiet, dzięki czemu trafiają do tych samych metryk.

Problem jest sprowadzony do **klasyfikacji jednoetykietowej (28 klas = 27 emocji + `neutral`)**: z GoEmotions zostawiamy przykłady z dokładnie jedną emocją.

---

## Instrukcja uruchomienia

**Środowisko:** Google Colab → `Runtime` → `Change runtime type` → **GPU (T4)**.

**Flagi sterujące:**
- `MODELS_DIR` — katalog na zapisane modele/wyniki
- `FORCE_RETRAIN` — `False`: jeśli model/wyniki są już zapisane na dysku, zostaną wczytane zamiast liczone od nowa; `True`: wymusza ponowny trening
- `GPT_EVAL_N` — liczba przykładów testowych ocenianych przez modele GPT (zero-shot jest wolny). `None` = cały zbiór testowy


## Konfiguracja środowiska

Instalacja zależności, import bibliotek, ustawienie ziarna losowości oraz flag sterujących.
Modele BERT są publiczne, a modele GPT ładujemy z **niebramkowanych** kopii (`unsloth/...`).

In [ ]:
# Instalacja zależności.
# - sentencepiece: wymagany przez tokenizery DeBERTa-v3 i ALBERT
# - bitsandbytes + accelerate: ładowanie modeli GPT w 4-bit na GPU
# - statsmodels: test McNemara (porównanie istotności różnic między modelami)
!pip install -q -U transformers datasets evaluate accelerate bitsandbytes sentencepiece scikit-learn seaborn statsmodels

In [ ]:
import os
import gc
import re
import json
import time
import random
from itertools import combinations

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
)
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    accuracy_score,
    f1_score,
)

SEED = 67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Urządzenie:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
FORCE_RETRAIN = False   # True => trenuj/licz od nowa, ignorując zapisane modele i wyniki
GPT_EVAL_N = 500        # liczba przykładów testowych dla modeli GPT; None => cały test

# --- Katalog na zapisane modele i wyniki ---
# Na Colab - Google Drive
# Lokalnie (poza Colab) używamy zwykłego folderu w bieżącym katalogu.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MODELS_DIR = "/content/drive/MyDrive/ssn_emotions_models"
except Exception:
    MODELS_DIR = "./ssn_emotions_models"

RESULTS_DIR = os.path.join(MODELS_DIR, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Katalog modeli/wyników:", MODELS_DIR)

## Wczytanie i przygotowanie danych (GoEmotions)

Wczytujemy konfigurację `simplified` zbioru GoEmotions (43 410 / 5 426 / 5 427 przykładów: train / validation / test).
GoEmotions jest **wieloetykietowy**, dlatego sprowadzamy go do problemu jednoetykietowego:

1. zostawiamy tylko przykłady z **dokładnie jedną** emocją,
2. wyciągamy tę emocję do pola `label`.

In [ ]:
dataset = load_dataset("go_emotions", "simplified")
print(dataset)

In [ ]:
# Filtr do przykładów jednoetykietowych + ekstrakcja pojedynczej etykiety do pola "label".
dataset = dataset.filter(lambda d: len(d["labels"]) == 1)
dataset = dataset.map(lambda d: {"label": d["labels"][0]})

print("Rozmiary splitów po filtracji:")
print({split: len(dataset[split]) for split in dataset})

In [ ]:
# Pełna lista 28 etykiet GoEmotions (kolejność = indeksy używane w polu "label").
emotions = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
NUM_LABELS = len(emotions)
id2label = {i: e for i, e in enumerate(emotions)}
label2id = {e: i for i, e in enumerate(emotions)}
print("Liczba klas:", NUM_LABELS)

In [ ]:
train_labels = np.array(dataset["train"]["label"])
counts = pd.Series(train_labels).value_counts().sort_index()
counts.index = [id2label[i] for i in counts.index]

plt.figure(figsize=(12, 7))
counts.sort_values().plot(kind="barh", color="steelblue")
plt.title("Rozkład klas w zbiorze treningowym (single-label GoEmotions)")
plt.xlabel("Liczba przykładów")
plt.tight_layout()
plt.show()

counts.sort_values(ascending=False)

## Wspólny moduł metryk i funkcje pomocnicze

Aby wyniki modeli BERT i GPT były w pełni porównywalne, definiujemy tu jeden zestaw funkcji liczących metryki.


Definiujemy też:
- `compute_metrics` — używane przez `Trainer` w trakcie treningu BERT (z `zero_division=0`, bo rzadkie klasy potrafią nie mieć predykcji),
- mapowania emocji na grupy sentymentu (positive/negative/ambiguous/neutral) do ewaluacji hierarchicznej,
- funkcje `save_results` / `load_results` do (de)serializacji wyników na dysk

In [ ]:
# Pogrupowanie emocji GoEmotions wg sentymentu
SENTIMENT_GROUPS = {
    "positive": ["admiration", "amusement", "approval", "caring", "desire", "excitement",
                 "gratitude", "joy", "love", "optimism", "pride", "relief"],
    "negative": ["anger", "annoyance", "disappointment", "disapproval", "disgust",
                 "embarrassment", "fear", "grief", "nervousness", "remorse", "sadness"],
    "ambiguous": ["confusion", "curiosity", "realization", "surprise"],
    "neutral": ["neutral"],
}
emotion_to_sentiment = {e: g for g, es in SENTIMENT_GROUPS.items() for e in es}
sentiment_names = list(SENTIMENT_GROUPS.keys())
sentiment_to_id = {s: i for i, s in enumerate(sentiment_names)}

# Wektor mapujący id emocji (0..27) -> id grupy sentymentu (0..3).
emo_id_to_sent_id = np.array(
    [sentiment_to_id[emotion_to_sentiment[id2label[i]]] for i in range(NUM_LABELS)]
)

In [ ]:
def basic_metrics(y_true, y_pred):
    """Podstawowe metryki klasyfikacji wspólne dla wszystkich modeli."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
    }


def compute_metrics(eval_pred):
    """Metryki liczone przez Trainer w trakcie treningu BERT (predykcja = argmax logitów)."""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }


def top_k_accuracy(y_true, proba, k=3):
    """Czy prawdziwa klasa jest wśród k najbardziej prawdopodobnych. Wymaga rozkładu prawdopodobienstw (tylko BERT)."""
    if proba is None:
        return None
    topk = np.argsort(proba, axis=1)[:, -k:]
    return float(np.mean([yt in row for yt, row in zip(np.asarray(y_true), topk)]))


def expected_calibration_error(y_true, proba, n_bins=10):
    """ECE: średnia rozbieżność między pewnością modelu a rzeczywistą trafnością. Tylko BERT (wymaga proba)."""
    if proba is None:
        return None
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == np.asarray(y_true)).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 0:
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

In [ ]:
# Serializacja wyników na dysk
def save_results(name, res):
    path = os.path.join(RESULTS_DIR, f"{name}.npz")
    np.savez_compressed(
        path,
        true=np.asarray(res["true"]),
        pred=np.asarray(res["pred"]),
        proba=res["proba"] if res["proba"] is not None else np.array([]),
        metrics=json.dumps(res["metrics"]),
        extra=json.dumps(res.get("extra", {})),
    )


def load_results(name):
    path = os.path.join(RESULTS_DIR, f"{name}.npz")
    if not os.path.exists(path):
        return None
    d = np.load(path, allow_pickle=True)
    proba = d["proba"]
    return {
        "true": d["true"],
        "pred": d["pred"],
        "proba": None if proba.size == 0 else proba,
        "metrics": json.loads(str(d["metrics"])),
        "extra": json.loads(str(d["extra"])),
    }


# Globalny słownik z wynikami wszystkich modeli:
#   results[name] = {"true", "pred", "proba" (lub None), "metrics", "extra"}
results = {}

## Modele BERT (fine-tuning)

Trenujemy trzy klasyfikatory 28-klasowe na **pełnym** zbiorze treningowym:

| Klucz | Model |
|------|-------|
| `deberta` | `microsoft/deberta-v3-base` |
| `roberta` | `roberta-base` |
| `albert`  | `albert-base-v2` |

Funkcja `load_or_train_bert`:
- jeśli model jest już zapisany w `MODELS_DIR` i `FORCE_RETRAIN=False` → wczytuje go z dysku i wykonuje tylko predykcję na teście,
- w przeciwnym razie → trenuje (3 epoki, lr 2e-5, `load_best_model_at_end` wg macro F1), zapisuje model i tokenizer na dysk, a następnie przewiduje.

Dla każdego modelu zapisujemy do `results`: prawdziwe etykiety, predykcje (`argmax`), rozkład prawdopodobieństwa (`softmax`) oraz latencję predykcji.

In [ ]:
BERT_MODELS = {
    "deberta": "microsoft/deberta-v3-base",
    "roberta": "roberta-base",
    "albert": "albert-base-v2",
}

In [ ]:
def load_or_train_bert(name, model_id):
    """Trenuje (lub wczytuje z dysku) klasyfikator BERT i zwraca predykcje na zbiorze testowym
    w jednolitym formacie zgodnym ze słownikiem `results`."""
    out_dir = os.path.join(MODELS_DIR, name)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Tokenizacja całego zbioru
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

    tokenized = dataset.map(tokenize_function, batched=True)

    saved = os.path.isdir(out_dir) and os.path.exists(os.path.join(out_dir, "config.json"))

    if saved and not FORCE_RETRAIN:
        print(f"[{name}] Wczytywanie zapisanego modelu z: {out_dir}")
        model = AutoModelForSequenceClassification.from_pretrained(out_dir)
    else:
        print(f"[{name}] Trening modelu: {model_id}")
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
        )
        training_args = TrainingArguments(
            output_dir=os.path.join(out_dir, "checkpoints"),
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=3,
            weight_decay=0.01,
            logging_steps=100,
            fp16=(device == "cuda"),
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            save_total_limit=1,
            report_to="none",
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized["train"],
            eval_dataset=tokenized["validation"],
            compute_metrics=compute_metrics,
        )
        trainer.train()
        # Zapis najlepszego modelu i tokenizera na dysk
        model.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        print(f"[{name}] Zapisano model w: {out_dir}")

    # Predykcja na zbiorze testowym + pomiar latencji (czas na pojedynczy przykład).
    pred_args = TrainingArguments(
        output_dir=os.path.join(out_dir, "tmp_predict"),
        per_device_eval_batch_size=32,
        fp16=(device == "cuda"),
        report_to="none",
    )
    predictor = Trainer(model=model, args=pred_args)
    t0 = time.time()
    pred_out = predictor.predict(tokenized["test"])
    latency = (time.time() - t0) / len(tokenized["test"])

    logits = torch.tensor(pred_out.predictions)
    proba = torch.softmax(logits, dim=-1).numpy()
    preds = proba.argmax(axis=-1)
    y_true = np.array(tokenized["test"]["label"])

    res = {
        "true": y_true,
        "pred": preds,
        "proba": proba,
        "metrics": basic_metrics(y_true, preds),
        "extra": {"latency_per_sample_s": latency, "type": "bert"},
    }

    del model, predictor
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return res

In [ ]:
# Pętla po modelach BERT. Przy FORCE_RETRAIN=False najpierw próbujemy wczytać gotowe wyniki z dysku.
for name, model_id in BERT_MODELS.items():
    cached = None if FORCE_RETRAIN else load_results(name)
    if cached is not None:
        print(f"[{name}] Wczytano zapisane wyniki z dysku.")
        results[name] = cached
    else:
        results[name] = load_or_train_bert(name, model_id)
        save_results(name, results[name])
    print(f"[{name}] metryki:", results[name]["metrics"])

## Modele GPT (zero-shot)

Modele generatywne nie są trenowane - dostają w prompcie listę 28 dozwolonych etykiet i mają zwrócić dokładnie jedną z nich. Odpowiedź mapujemy na id klasy funkcją `parse_emotion`, dzięki czemu wyniki trafiają do tych samych metryk co BERT.

**Modele (niebramkowane kopie 4-bit, ładowane bez tokenu HF):**

| Klucz | Model |
|------|-------|
| `llama3` | `unsloth/llama-3-8b-Instruct-bnb-4bit` |
| `gemma2` | `unsloth/gemma-2-9b-it-bnb-4bit` |

Uwagi:
- zero-shot na całym teście jest wolny — `GPT_EVAL_N` ogranicza liczbę przykładów (domyślnie 500),
- przy braku pamięci GPU można podmienić Gemmę na `unsloth/gemma-2-2b-it-bnb-4bit`,
- modele GPT nie zwracają rozkładu prawdopodobieństwa, więc `proba=None` (metryki top-k / ECE będą dla nich N/A),
- zapisujemy też `no_match_rate` — odsetek odpowiedzi, których nie udało się zmapować na żadną etykietę (fallback = `neutral`).

> Uwaga metodologiczna: modele BERT oceniamy na pełnym teście, a GPT na podzbiorze `GPT_EVAL_N` (pierwsze N przykładów testu). Metryki każdego modelu liczone są na jego własnych danych; porównania bezpośrednie (sekcja 5.9) zawężamy do wspólnego podzbioru.

In [ ]:
GPT_MODELS = {
    "llama3": "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "gemma2": "unsloth/gemma-2-9b-it-bnb-4bit",
}

# Instrukcja zero-shot: pełna lista 28 etykiet + wymóg zwrócenia dokładnie jednej z nich.
EMOTION_LIST_STR = ", ".join(emotions)
PROMPT_TEMPLATE = (
    "You are an emotion classification system.\n"
    "Classify the single dominant emotion expressed in the text below.\n"
    "Answer with EXACTLY ONE label from this list and nothing else:\n"
    f"{EMOTION_LIST_STR}.\n\n"
    'Text: "{text}"\n'
    "Emotion:"
)


def parse_emotion(generated_text):
    """Mapuje swobodną odpowiedź modelu na id jednej z 28 emocji.
    Zwraca (label_id, matched); matched=False oznacza brak trafienia -> fallback = neutral."""
    text = generated_text.lower().strip()
    # Dopasowanie całego słowa będącego nazwą emocji (pierwsze znalezione).
    for i, emo in enumerate(emotions):
        if re.search(rf"\b{re.escape(emo)}\b", text):
            return i, True
    return label2id["neutral"], False

In [ ]:
@torch.no_grad()
def classify_with_gpt(model_id, texts):
    """Klasyfikacja zero-shot listą tekstów. Zwraca (predykcje, liczba_nietrafień, latencja_na_przykład)."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.float16,
        attn_implementation="eager",
    )
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    preds, n_unmatched = [], 0
    t0 = time.time()
    for text in texts:
        prompt = PROMPT_TEMPLATE.format(text=text)
        messages = [{"role": "user", "content": prompt}]
        # Chat template dostosowuje prompt do formatu danego modelu instruct.
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        out = model.generate(
            inputs,
            max_new_tokens=8,          # wystarczy na pojedynczą etykietę
            do_sample=False,           # deterministycznie (greedy) dla powtarzalności
            pad_token_id=tokenizer.pad_token_id,
        )
        generated = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        label_id, matched = parse_emotion(generated)
        preds.append(label_id)
        if not matched:
            n_unmatched += 1
    latency = (time.time() - t0) / max(len(texts), 1)

    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return np.array(preds), n_unmatched, latency

In [ ]:
# Bierzemy pierwsze GPT_EVAL_N przykładów testu (lub cały test, gdy GPT_EVAL_N=None).
test_texts_full = dataset["test"]["text"]
test_labels_full = np.array(dataset["test"]["label"])

n_gpt = len(test_texts_full) if GPT_EVAL_N is None else min(GPT_EVAL_N, len(test_texts_full))
gpt_idx = np.arange(n_gpt)
gpt_texts = [test_texts_full[i] for i in gpt_idx]
gpt_true = test_labels_full[gpt_idx]
print(f"Liczba przykładów do oceny przez modele GPT: {n_gpt}")

for name, model_id in GPT_MODELS.items():
    cached = None if FORCE_RETRAIN else load_results(name)
    if cached is not None:
        print(f"[{name}] Wczytano zapisane wyniki z dysku.")
        results[name] = cached
        continue

    print(f"[{name}] Zero-shot klasyfikacja {n_gpt} przykładów modelem: {model_id}")
    preds, n_unmatched, latency = classify_with_gpt(model_id, gpt_texts)
    results[name] = {
        "true": gpt_true,
        "pred": preds,
        "proba": None,
        "metrics": basic_metrics(gpt_true, preds),
        "extra": {
            "latency_per_sample_s": latency,
            "no_match_rate": n_unmatched / max(n_gpt, 1),
            "type": "gpt",
        },
    }
    save_results(name, results[name])
    print(f"[{name}] metryki:", results[name]["metrics"],
          "| no_match_rate:", round(results[name]["extra"]["no_match_rate"], 4))

## Metryki i ewaluacja

Wszystkie modele (BERT i GPT) trafiają do tego samego zestawu metryk. Każda metryka ma osobny opis i osobną komórkę z kodem, działającą w pętli po `results`.

### Tabela zbiorcza

Zestawienie podstawowych metryk dla wszystkich modeli: **accuracy**, **macro F1** (traktuje klasy równo), **weighted F1** (waży klasy ich licznością) oraz **Cohen's Kappa** (zgodność skorygowana o przypadek). Sortujemy po macro F1.

In [ ]:
summary = pd.DataFrame({name: r["metrics"] for name, r in results.items()}).T
summary = summary[["accuracy", "f1_macro", "f1_weighted", "cohen_kappa"]]
summary["type"] = [results[n]["extra"].get("type", "") for n in summary.index]
summary["latency_s/ex"] = [results[n]["extra"].get("latency_per_sample_s", np.nan) for n in summary.index]
summary.sort_values("f1_macro", ascending=False)

### Macierz pomyłek (confusion matrix)

Macierz 28×28 dla każdego modelu. Wiersze = klasa prawdziwa, kolumny = przewidziana. Wartości znormalizowane wierszami (udział w danej klasie prawdziwej), co uniezależnia odczyt od imbalansu. Jasna przekątna = dobre dopasowanie; jasne pola poza przekątną wskazują systematyczne pomyłki.

In [ ]:
def plot_confusion(name):
    r = results[name]
    cm = confusion_matrix(r["true"], r["pred"], labels=range(NUM_LABELS))
    # Normalizacja wierszami (udział w klasie prawdziwej); zabezpieczenie przed dzieleniem przez 0.
    cm_norm = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    plt.figure(figsize=(14, 11))
    sns.heatmap(cm_norm, cmap="Blues", xticklabels=emotions, yticklabels=emotions, vmin=0, vmax=1)
    plt.title(f"Macierz pomyłek (znormalizowana wierszami) - {name}")
    plt.xlabel("Przewidziana emocja")
    plt.ylabel("Prawdziwa emocja")
    plt.tight_layout()
    plt.show()


for name in results:
    plot_confusion(name)

### F1-score per emocja

Wykres F1 dla każdej z 28 emocji osobno, z zaznaczoną średnią (macro F1). Pokazuje, które emocje są łatwe (np. `gratitude`, `love`), a które trudne (rzadkie/wieloznaczne, np. `grief`, `realization`, `nervousness`). Czerwona linia = macro F1 danego modelu.

In [ ]:
def plot_f1_per_class(name):
    r = results[name]
    report = classification_report(
        r["true"], r["pred"], labels=range(NUM_LABELS),
        target_names=emotions, output_dict=True, zero_division=0,
    )
    f1s = pd.Series({e: report[e]["f1-score"] for e in emotions})
    plt.figure(figsize=(10, 10))
    f1s.sort_values().plot(kind="barh", color="skyblue")
    plt.axvline(f1s.mean(), color="red", linestyle="--", label=f"macro F1 = {f1s.mean():.3f}")
    plt.title(f"F1-score per emocja - {name}")
    plt.xlabel("F1-score")
    plt.legend()
    plt.tight_layout()
    plt.show()


for name in results:
    plot_f1_per_class(name)

### Accuracy i pełny classification_report

Szczegółowy raport (precision, recall, F1, support) per klasa dla każdego modelu. `zero_division=0` zapobiega błędom dla klas bez predykcji (częste przy rzadkich emocjach).

In [ ]:
for name, r in results.items():
    print("=" * 70)
    print(f"Model: {name}  |  accuracy = {r['metrics']['accuracy']:.4f}")
    print("=" * 70)
    print(classification_report(
        r["true"], r["pred"], labels=range(NUM_LABELS),
        target_names=emotions, zero_division=0,
    ))

### Cohen's Kappa

Kappa mierzy zgodność predykcji z prawdą **skorygowaną o zgodność losową**. Jest bardziej wymagająca niż accuracy przy imbalansie (zgadywanie klasy większościowej nie podbija wyniku). Im bliżej 1.0, tym lepiej; ~0 oznacza poziom losowy.

In [ ]:
kappa_tbl = pd.Series(
    {name: cohen_kappa_score(r["true"], r["pred"]) for name, r in results.items()}
).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
kappa_tbl.plot(kind="bar", color="teal")
plt.title("Cohen's Kappa per model")
plt.ylabel("kappa")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

kappa_tbl

### Ewaluacja hierarchiczna (grupy sentymentu)

28 szczegółowych emocji mapujemy na 4 grupy sentymentu GoEmotions: **positive / negative / ambiguous / neutral**. Wiele pomyłek to emocje "blisko siebie" (np. `annoyance` ↔ `anger`, `joy` ↔ `excitement`) — na zgrubnym poziomie model może być znacznie lepszy. Porównanie metryk 28-klasowych z 4-grupowymi pokazuje, ile błędów to drobne pomyłki w obrębie tej samej grupy.

In [ ]:
def hierarchical_metrics(name):
    r = results[name]
    # Rzutowanie etykiet 28-klasowych na 4 grupy sentymentu.
    yt = emo_id_to_sent_id[np.asarray(r["true"])]
    yp = emo_id_to_sent_id[np.asarray(r["pred"])]
    return {
        "acc_28": r["metrics"]["accuracy"],
        "f1_macro_28": r["metrics"]["f1_macro"],
        "acc_sentiment": accuracy_score(yt, yp),
        "f1_macro_sentiment": f1_score(yt, yp, average="macro", zero_division=0),
    }


hier = pd.DataFrame({name: hierarchical_metrics(name) for name in results}).T
hier.sort_values("f1_macro_sentiment", ascending=False)

### Top-k accuracy (k=3)

Czy prawdziwa emocja znajduje się wśród 3 najpewniejszych predykcji modelu. Emocje bywają wieloznaczne, więc top-1 potrafi być zbyt surowe. Metryka wymaga rozkładu prawdopodobieństwa - dostępna tylko dla modeli BERT (dla GPT zwraca `None`).

In [ ]:
topk_tbl = pd.DataFrame({
    name: {
        "top1_accuracy": r["metrics"]["accuracy"],
        "top3_accuracy": top_k_accuracy(r["true"], r["proba"], k=3),
    }
    for name, r in results.items()
}).T
# top3 = NaN dla modeli GPT (brak rozkładu prawdopodobieństwa).
topk_tbl

### Najczęściej mylone pary emocji

Ranking największych wartości poza przekątną macierzy pomyłek — pokazuje, które emocje model najczęściej myli (np. `approval` → `neutral`). Pomaga zinterpretować błędy jako sensowne (bliskie emocje) lub przypadkowe.

In [ ]:
def most_confused_pairs(name, top_n=10):
    r = results[name]
    cm = confusion_matrix(r["true"], r["pred"], labels=range(NUM_LABELS))
    pairs = [
        (emotions[i], emotions[j], int(cm[i, j]))
        for i in range(NUM_LABELS)
        for j in range(NUM_LABELS)
        if i != j and cm[i, j] > 0
    ]
    df = pd.DataFrame(pairs, columns=["prawdziwa", "przewidziana", "liczba"])
    return df.sort_values("liczba", ascending=False).head(top_n).reset_index(drop=True)


for name in results:
    print(f"--- {name}: top mylonych par (prawdziwa -> przewidziana) ---")
    print(most_confused_pairs(name).to_string(index=False))
    print()

### Zgodność między modelami + test McNemara

- **Pairwise Cohen's Kappa** - jak bardzo modele zgadzają się ze sobą (niezależnie od poprawności). Niska zgodność BERT↔GPT sugeruje, że popełniają różne błędy.
- **Test McNemara** - czy różnica trafności dwóch modeli jest istotna statystycznie (p < 0.05). Liczony na wspólnym podzbiorze przykładów (pierwsze `n` testu, bo GPT oceniano na podzbiorze).

In [ ]:
# Wspólny podzbiór: predykcje wszystkich modeli na tych samych przykładach.
# BERT przewiduje cały test, GPT pierwsze n - bierzemy więc pierwsze common_n predykcji każdego modelu
# (kolejność testu jest stała, więc indeksy się zgadzają).
names = list(results.keys())
common_n = min(len(np.asarray(r["pred"])) for r in results.values())
aligned = {name: np.asarray(results[name]["pred"])[:common_n] for name in names}
common_true = np.asarray(results[names[0]]["true"])[:common_n]

# Macierz pairwise Cohen's Kappa między modelami.
agree = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        agree.loc[a, b] = cohen_kappa_score(aligned[a], aligned[b])

plt.figure(figsize=(7, 6))
sns.heatmap(agree.astype(float), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.title(f"Zgodność między modelami - Cohen's Kappa (n={common_n})")
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

# Test McNemara dla każdej pary modeli na wspólnym podzbiorze.
rows = []
for a, b in combinations(names, 2):
    correct_a = aligned[a] == common_true
    correct_b = aligned[b] == common_true
    n_a_only = int(np.sum(correct_a & ~correct_b))   # tylko A trafił
    n_b_only = int(np.sum(~correct_a & correct_b))   # tylko B trafił
    table = [[0, n_a_only], [n_b_only, 0]]
    p_value = mcnemar(table, exact=False, correction=True).pvalue
    rows.append((a, b, n_a_only, n_b_only, p_value, "tak" if p_value < 0.05 else "nie"))

mcnemar_df = pd.DataFrame(
    rows, columns=["model_A", "model_B", "tylko_A_trafil", "tylko_B_trafil", "p_value", "istotne(p<0.05)"]
)
mcnemar_df

### Jakość vs koszt (latencja)

Zestawienie jakości (macro F1) z kosztem (latencja predykcji na przykład). Fine-tunowane BERT-y są małe i szybkie, ale wymagały treningu; modele GPT działają zero-shot, lecz są znacznie wolniejsze i cięższe. Wykres pomaga ocenić, co opłaca się w praktyce.

In [ ]:
lat = pd.Series({name: r["extra"].get("latency_per_sample_s", np.nan) for name, r in results.items()})
f1m = pd.Series({name: r["metrics"]["f1_macro"] for name, r in results.items()})

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(lat.index, lat.values, color="salmon", alpha=0.7)
ax1.set_ylabel("Latencja [s / przykład]", color="salmon")
ax1.tick_params(axis="y", labelcolor="salmon")

ax2 = ax1.twinx()
ax2.plot(f1m.index, f1m.values, color="navy", marker="o", linewidth=2)
ax2.set_ylabel("macro F1", color="navy")
ax2.tick_params(axis="y", labelcolor="navy")

plt.title("Trade-off: jakość (macro F1) vs koszt (latencja)")
fig.tight_layout()
plt.show()

pd.DataFrame({"macro_f1": f1m, "latency_s/ex": lat}).sort_values("macro_f1", ascending=False)

### F1 per emocja vs liczność klasy

Wykres punktowy: oś X = liczba przykładów treningowych danej emocji (skala log), oś Y = F1 tej emocji. Pokazuje wpływ imbalansu — czy słabe wyniki dotyczą głównie rzadkich klas. Modele zero-shot GPT powinny być mniej zależne od liczności (nie uczyły się na tych danych).

In [ ]:
# Liczność każdej klasy w zbiorze treningowym.
train_counts = pd.Series(np.array(dataset["train"]["label"])).value_counts()
class_counts = np.array([train_counts.get(i, 0) for i in range(NUM_LABELS)])

plt.figure(figsize=(9, 6))
for name, r in results.items():
    report = classification_report(
        r["true"], r["pred"], labels=range(NUM_LABELS),
        target_names=emotions, output_dict=True, zero_division=0,
    )
    f1s = np.array([report[e]["f1-score"] for e in emotions])
    plt.scatter(class_counts, f1s, label=name, alpha=0.7)

plt.xscale("log")
plt.xlabel("Liczba przykładów treningowych klasy (log)")
plt.ylabel("F1-score klasy")
plt.title("F1 per emocja vs liczność klasy")
plt.legend()
plt.tight_layout()
plt.show()

## Podsumowanie i wnioski

Poniższa komórka generuje końcowy ranking modeli wg macro F1.

In [ ]:
# Końcowy ranking modeli wg macro F1.
final = pd.DataFrame({name: r["metrics"] for name, r in results.items()}).T
final["type"] = [results[n]["extra"].get("type", "") for n in final.index]
final = final.sort_values("f1_macro", ascending=False)
print("Ranking modeli wg macro F1:")
print(final.to_string())

best = final.index[0]
print(f"\nNajlepszy model: {best} (macro F1 = {final.loc[best, 'f1_macro']:.4f})")